# Stap 4 - hoekmeting en PID
Gebruik `voer_pid_stap4_uit` met `Stap_4_PID.ino`. Hieronder stel je de PID af.

Verderop staat **stap 5: PWM-veranderingsbegrenzing**. Daar gebruik je `voer_pid_stap5_uit` met `Stap_5_PWM_begrenzing.ino`.


In [1]:
from closed_loop.meting import test_hoek, voer_pid_stap4_uit, voer_pid_stap5_uit
from closed_loop.figuren import plot_pid_meting
import matplotlib.pyplot as plt
com_poort = "COM4"  # Vul de poort van jouw Arduino in.


### Ruwe hoek meten

Laat de motorvoeding uit. Geef meetduur en delta_t in seconden op. De uitvoer toont Arduino-tijd vanaf het eerste meetpunt en de ruwe sensorhoek. PWM is nul. De meting wordt als CSV opgeslagen in data/. Noteer de hoek in de nulstand; de correctie voegen we in de volgende stap toe.


In [2]:
hoekmetingen = test_hoek(com_poort=com_poort, meetduur=1.0, delta_t=0.1)


PWM: 0; stabiliseren: 0 s
Meting gestart. Onderbreek de notebookcel om te stoppen.
   0.000 s |  115.137 graden
   0.100 s |  115.137 graden
   0.200 s |  115.137 graden
   0.300 s |  115.137 graden
   0.400 s |  115.137 graden
   0.500 s |  115.137 graden
   0.600 s |  115.137 graden
   0.700 s |  115.137 graden
   0.800 s |  115.137 graden
   0.900 s |  115.137 graden
STOP verzonden naar de Arduino.
CSV opgeslagen: C:\Users\jbusink1\OneDrive - HvA\Documenten\GitHub\Practica HvA Github\practicum2_2\Closed_loop\data\aeropendulum_20261112_113650_340480.csv
Aantal metingen: 10


### 2. PID instellen en meten

Begin met ki = 0 en kd = 0. Vul de gemeten beginhoek in. De werkhoek is sensorhoek - beginhoek (omgerekend naar -180..180). Controleer dat de werkhoek toeneemt als de motor de arm omhoog beweegt. De PID berekent de PWM; max_pwm begrenst deze van 0 tot en met 255 (0 houdt de motor uit). Doelhoek mag van -180 tot 180 graden zijn, gemeten vanaf stilstand. Bij beginhoek 89 en sensorhoek 89 is de werkhoek 0; bij sensorhoek 0 is de werkhoek 89 graden, ongeveer 90. Een doelhoek van 90 betekent een verplaatsing van 90 graden vanaf stilstand, niet een ruwe sensorwaarde van 90. Met toon_metingen kies je of de meetpunten live worden geprint. Fixed stopt na meetduur; continuous loopt tot je de cel onderbreekt. Delta T bepaalt het meet- en regelinterval.


In [3]:
# Stap 4: upload Stap_4_PID.ino.
beginhoek = 115.4  # Vervang door de gemeten sensorhoek in de nulstand.
doelhoek = 90.0    # graden ten opzichte van de ingestelde nulstand
kp = 3
ki = 3
kd = 3
max_pwm = 255     # bovengrens; kies een lagere waarde indien nodig
meetduur = 10.0   # seconden; stoptijd in fixed mode
delta_t = 0.02    # seconden (0.02-1); frequentie = 1 / delta_t Hz
mode = "fixed"   # "fixed" of "continuous" (stop met cel onderbreken)

toon_metingen = True  # False: geen meetpunten printen

metingen, csv_pad = voer_pid_stap4_uit(
    com_poort=com_poort, beginhoek=beginhoek,
    doelhoek=doelhoek,
    kp=kp, ki=ki, kd=kd,
    max_pwm=max_pwm, toon_metingen=toon_metingen,
    meetduur=meetduur, delta_t=delta_t, mode=mode
)

Stap 4: max_pwm=255, limiter=uit telwaarden/regelstap


RuntimeError: Verkeerde of verouderde sketch: stap 4 verwacht Stap_4_PID.ino. Upload de bijgewerkte sketch. De motor is niet gestart.

### 3. Resultaat bekijken en CSV inlezen

De PID-meting wordt opgeslagen in een CSV met alle instellingen bovenaan (regels die met `#` beginnen), gevolgd door `tijd_s,hoek_deg,pwm`. Er wordt geen apart JSON-bestand meer aangemaakt. Tijden in de instellingen met `_ms` zijn in milliseconden; `tijd_s` in de meetdata is in seconden.

De cel hieronder leest de laatst opgeslagen PID-meting in. Vervang `csv_pad` door een bestandspad om een andere meting te openen. Voor een veiligheidsproef gebruik je `csv_pad_veiligheid`. Deze uitleescel is bedoeld voor het nieuwe CSV-formaat.


In [ ]:
import csv
import pandas as pd

bestand = csv_pad  # Of bijvoorbeeld "data/pid_20261027_....csv"

# Lees de instellingen uit de commentaarheader.
with open(bestand, encoding="utf-8", newline="") as f:
    instellingen = {}
    for rij in csv.reader(f):
        if not rij or not rij[0].startswith("# "):
            break
        instellingen[rij[0][2:]] = rij[1]

# Pandas slaat de commentaarheader over en leest de meetkolommen.
df = pd.read_csv(bestand, comment="#")
display(pd.Series(instellingen, name="Instelling").to_frame())
display(df.head())


### 4. De maximale hoek aanpassen

Gebruik dezelfde functie en dezelfde geuploade sketch. Tot nu toe hoefde je geen `max_hoek` mee te geven: de functie gebruikt standaard 180 graden. Geef nu `max_hoek=120` mee. Bij overschrijding van de ingestelde werkhoek stopt de Arduino de motor en de meting. De doelhoek moet binnen de ingestelde grens liggen.


In [4]:
# Vervolg: dezelfde sketch en functie, nu met een lagere stopgrens.
beginhoek = 118.4  # Vervang door de gemeten sensorhoek in de nulstand.
doelhoek = 90.0    # graden ten opzichte van de ingestelde nulstand
kp = 4
ki = 3
kd = 0
max_pwm = 255     # bovengrens; kies een lagere waarde indien nodig
meetduur = 10.0   # seconden; stoptijd in fixed mode
delta_t = 0.02    # seconden (0.02-1); frequentie = 1 / delta_t Hz
mode = "fixed"   # "fixed" of "continuous" (stop met cel onderbreken)

toon_metingen = True 
max_hoek = 100.0  # Stopgrens voor de werkhoek; kies passend bij de opstelling.

metingen_veiligheid, csv_pad_veiligheid = voer_pid_stap4_uit(
    com_poort=com_poort, beginhoek=beginhoek,
    doelhoek=doelhoek, max_hoek=max_hoek,
    kp=kp, ki=ki, kd=kd,
    max_pwm=max_pwm, toon_metingen=toon_metingen,
    meetduur=meetduur, delta_t=delta_t, mode=mode,
)


Stap 4: max_pwm=255, limiter=uit telwaarden/regelstap


RuntimeError: Verkeerde of verouderde sketch: stap 4 verwacht Stap_4_PID.ino. Upload de bijgewerkte sketch. De motor is niet gestart.

## Stap 5 - maximale PWM-sprong per regelstap

Gebruik `voer_pid_stap5_uit`. Zonder `max_pwm_stap` wordt geen begrenzer ingesteld. Met `max_pwm_stap=20` mag de werkelijk aangeboden PWM per regelstap maximaal 20 telwaarden stijgen; dalen gaat direct. Kleine veranderingen worden direct overgenomen. Een vraag van 0 naar 255 wordt bijvoorbeeld 0, 20, 40, 60, enzovoort. STOP en andere stops zetten de uitgang direct op nul.

Bij een ander regelinterval verandert ook de maximale verandering per seconde. `max_pwm` blijft de maximale uitgang; bij 0 draait de motor niet.

De begrenzer beperkt uitsluitend positieve PWM-sprongen. Een verlaging, ook naar nul, wordt direct overgenomen.


In [18]:
from closed_loop.meting import test_hoek, voer_pid_stap4_uit, voer_pid_stap5_uit
import pandas as pd
import matplotlib.pyplot as plt

instellingen = dict(
    com_poort="COM4",
    beginhoek=115.4,  # Vul je gemeten ruwe sensorhoek in rust in.
    doelhoek=90.0,
    kp=6.0, ki=4.0, kd=3.0,  # Neem je eigen stap-4-instelling over.
    max_pwm=255,  # Eerst communicatie testen; daarna je afgesproken PWM-grens.
    meetduur=10.0,
    delta_t=0.02,
    mode="fixed",
    toon_metingen=True,
)

metingen_zonder, csv_zonder = voer_pid_stap5_uit(
    **instellingen
)


Stap 5: max_pwm=255, limiter=uit telwaarden/regelstap
PID-meting gestart.
   0.020 s | hoek   -0.527 graden | doel   90.000 graden | PWM 255 (100.0%)
   0.040 s | hoek   -0.615 graden | doel   90.000 graden | PWM 255 (100.0%)
   0.060 s | hoek   -0.527 graden | doel   90.000 graden | PWM 255 (100.0%)
   0.080 s | hoek   -0.439 graden | doel   90.000 graden | PWM 255 (100.0%)
   0.100 s | hoek   -0.175 graden | doel   90.000 graden | PWM 255 (100.0%)
   0.120 s | hoek    0.176 graden | doel   90.000 graden | PWM 255 (100.0%)
   0.140 s | hoek    1.143 graden | doel   90.000 graden | PWM 255 (100.0%)
   0.160 s | hoek    2.286 graden | doel   90.000 graden | PWM 255 (100.0%)
   0.180 s | hoek    4.043 graden | doel   90.000 graden | PWM 255 (100.0%)
   0.200 s | hoek    6.329 graden | doel   90.000 graden | PWM 230 ( 90.2%)
   0.220 s | hoek    9.053 graden | doel   90.000 graden | PWM 155 ( 60.8%)
   0.240 s | hoek   12.481 graden | doel   90.000 graden | PWM  35 ( 13.7%)
   0.260 s | h

In [17]:
max_pwm_stap = 16
metingen_met, csv_met = voer_pid_stap5_uit(
    **instellingen, max_pwm_stap=max_pwm_stap,
)


Stap 5: max_pwm=255, limiter=16 telwaarden/regelstap
PID-meting gestart.
   0.020 s | hoek   38.409 graden | doel   90.000 graden | PWM  16 (  6.3%)
   0.040 s | hoek   38.584 graden | doel   90.000 graden | PWM  32 ( 12.5%)
   0.060 s | hoek   38.321 graden | doel   90.000 graden | PWM  48 ( 18.8%)
   0.080 s | hoek   37.530 graden | doel   90.000 graden | PWM  64 ( 25.1%)
   0.100 s | hoek   36.299 graden | doel   90.000 graden | PWM  80 ( 31.4%)
   0.120 s | hoek   34.541 graden | doel   90.000 graden | PWM  96 ( 37.6%)
   0.140 s | hoek   32.344 graden | doel   90.000 graden | PWM 112 ( 43.9%)
   0.160 s | hoek   29.707 graden | doel   90.000 graden | PWM 128 ( 50.2%)
   0.180 s | hoek   26.807 graden | doel   90.000 graden | PWM 144 ( 56.5%)
   0.200 s | hoek   23.379 graden | doel   90.000 graden | PWM 160 ( 62.7%)
   0.220 s | hoek   19.688 graden | doel   90.000 graden | PWM 176 ( 69.0%)
   0.240 s | hoek   15.469 graden | doel   90.000 graden | PWM 192 ( 75.3%)
   0.260 s | ho